<a href="https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
### Data contract

**Unit of analysis:** One row represents one client-content pair on one report date.

**Table(s):** The primary source is `fact_content_daily_performance`. The dimension tables are not required for the five-feature development frame.

**Time window:** March 2026 is the development/decision month. Historical daily observations before each decision date are used for features, while the following 7 days provide the forward outcome.

**Prediction/ranking target:** The target is `future_7d_gsc_clicks`, defined as the total GSC clicks for the same client-content pair during the 7 days after the decision date.

**Deliberate exclusion:** Future-period performance metrics, including the clicks used to construct `future_7d_gsc_clicks`, are excluded from the feature set because they are unavailable at the decision moment and would create label leakage.

In [44]:
# FlyRank ML-04 — Warehouse setup

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is not available. Check Colab Secrets."

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [45]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(hf_token))

HF_TOKEN loaded: True


In [3]:
%pip -q install duckdb

In [14]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


In [5]:
userdata.get("HF_TOKEN")

'HFAKtxTocFQmjkUEeinVDjysr97cPrw'

In [12]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

me = api.whoami()

print("Hugging Face authentication: OK")
print("Username:", me["name"])

Hugging Face authentication: OK
Username: Umair01code


In [13]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset access: OK")
print("Dataset:", info.id)

Dataset access: OK
Dataset: FlyRank/internship-warehouse


In [15]:
daily = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

march_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {daily}
""").df()

march_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [16]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {daily}
    LIMIT 1
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [17]:
sample_rows = con.sql(f"""
    SELECT *
    FROM {daily}
    LIMIT 5
""").df()

sample_rows

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [18]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS distinct_grain_rows
    FROM {daily}
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,9841378,9841378


In [19]:
duplicate_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM {daily}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    ORDER BY rows_per_key DESC
    LIMIT 10
""").df()

duplicate_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_per_key


In [20]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {daily}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [21]:
missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_gsc_clicks,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_gsc_avg_position,
        COUNT(*) FILTER (WHERE ga4_pageviews IS NULL) AS missing_ga4_pageviews,
        COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS missing_ga4_sessions,
        COUNT(*) FILTER (WHERE sessions_organic IS NULL) AS missing_sessions_organic
    FROM {daily}
""").df()

missing_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_ga4_pageviews,missing_ga4_sessions,missing_sessions_organic
0,9841378,0,0,6230317,3018741,3018741,3018741


In [24]:
months = con.sql("""
    SELECT DISTINCT month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_*.parquet',
        hive_partitioning = true
    )
    ORDER BY month
""").df()

months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


In [25]:
april_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
""").df()

april_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,10424730,2026-04-01,2026-04-30


In [26]:
feature_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available,

        COUNT(*) FILTER (
            WHERE sessions_organic IS NOT NULL
        ) AS organic_sessions_available,

        COUNT(*) FILTER (
            WHERE scroll_events IS NOT NULL
        ) AS scroll_events_available
    FROM {daily}
""").df()

feature_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available,organic_sessions_available,scroll_events_available
0,9841378,3611061,413966,6822637,6822637


## 2. Fields: feature / label / context / excluded
### Fields: feature / label / context / excluded

| Bucket | Field | Why |
|---|---|---|
| Feature | `prev_7d_gsc_clicks` | Historical GSC clicks from the 7 days before the decision date; available before the prediction is made. |
| Feature | `prev_7d_gsc_impressions` | Historical GSC impressions from the 7 days before the decision date; available before the prediction is made. |
| Feature | `prev_7d_avg_position` | Historical GSC average position over the prior 7 days; available before the prediction is made. |
| Feature | `prev_7d_organic_sessions` | Historical organic sessions over the prior 7 days; available before the prediction is made when the source is available. |
| Feature | `prev_7d_scroll_events` | Historical scroll events over the prior 7 days; available before the prediction is made when the source is available. |
| Label | `future_7d_gsc_clicks` | Total GSC clicks during the 7 days after the decision date; this is the outcome being predicted. |
| Context | `report_date` | Defines the decision date and temporal window; not used as a predictive feature. |
| Context | `client_hash_id` | Identifies the client-content grouping; retained for analysis but not used directly as a predictive feature. |
| Context | `content_hash_id` | Identifies the content item; retained for analysis but not used directly as a predictive feature. |
| Context | `gsc_data_available` | Indicates whether GSC data is available for the row. |
| Context | `ga4_data_available` | Indicates whether GA4 data is available for the row. |
| Excluded | Future-period performance fields used to construct the label | These are not knowable at the decision moment and would leak the outcome into the features. |

In [28]:
feature_frame.shape

(9841378, 9)

In [29]:
feature_frame.head()

,report_date,client_hash_id,content_hash_id,prev_7d_gsc_clicks,prev_7d_gsc_impressions,prev_7d_avg_position,prev_7d_organic_sessions,prev_7d_scroll_events,future_7d_gsc_clicks
0,2026-03-01,client_e547b89c05043229,content_89d4bfd4558ec9b9,0.0,117.0,8.223562,0.0,0.0,1.0
1,2026-03-01,client_e547b89c05043229,content_0f7da309daf39bb8,0.0,851.0,15.169861,0.0,0.0,1.0
2,2026-03-01,client_e547b89c05043229,content_8c99c861f5cb1a47,0.0,20.0,16.680556,0.0,0.0,0.0
3,2026-03-01,client_e547b89c05043229,content_6210e5a42848b5fb,5.0,2034.0,3.536536,2.0,0.0,1.0
4,2026-03-01,client_e547b89c05043229,content_8377b3ff6652698c,0.0,579.0,5.197947,0.0,0.0,1.0


In [30]:
feature_frame.isna().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
prev_7d_gsc_clicks,53618
prev_7d_gsc_impressions,53618
prev_7d_avg_position,5469503
prev_7d_organic_sessions,3113081
prev_7d_scroll_events,3113081
future_7d_gsc_clicks,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
feature_sql = """
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        sessions_organic,
        scroll_events
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    )

    UNION ALL

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        sessions_organic,
        scroll_events
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    UNION ALL

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        sessions_organic,
        scroll_events
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
),

march AS (
    SELECT DISTINCT
        report_date,
        client_hash_id,
        content_hash_id
    FROM base
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
),

features AS (
    SELECT
        m.report_date,
        m.client_hash_id,
        m.content_hash_id,

        SUM(b.gsc_clicks) AS prev_7d_gsc_clicks,
        SUM(b.gsc_impressions) AS prev_7d_gsc_impressions,
        AVG(b.gsc_avg_position) AS prev_7d_avg_position,
        SUM(b.sessions_organic) AS prev_7d_organic_sessions,
        SUM(b.scroll_events) AS prev_7d_scroll_events

    FROM march m
    LEFT JOIN base b
        ON b.client_hash_id = m.client_hash_id
        AND b.content_hash_id = m.content_hash_id
        AND b.report_date BETWEEN
            m.report_date - INTERVAL '7 days'
            AND m.report_date - INTERVAL '1 day'

    GROUP BY
        m.report_date,
        m.client_hash_id,
        m.content_hash_id
),

labels AS (
    SELECT
        m.report_date,
        m.client_hash_id,
        m.content_hash_id,
        SUM(b.gsc_clicks) AS future_7d_gsc_clicks

    FROM march m
    LEFT JOIN base b
        ON b.client_hash_id = m.client_hash_id
        AND b.content_hash_id = m.content_hash_id
        AND b.report_date BETWEEN
            m.report_date + INTERVAL '1 day'
            AND m.report_date + INTERVAL '7 days'

    GROUP BY
        m.report_date,
        m.client_hash_id,
        m.content_hash_id
)

SELECT
    f.*,
    l.future_7d_gsc_clicks
FROM features f
LEFT JOIN labels l
    ON f.report_date = l.report_date
    AND f.client_hash_id = l.client_hash_id
    AND f.content_hash_id = l.content_hash_id
"""

feature_frame = con.sql(feature_sql).df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,prev_7d_gsc_clicks,prev_7d_gsc_impressions,prev_7d_avg_position,prev_7d_organic_sessions,prev_7d_scroll_events,future_7d_gsc_clicks
0,2026-03-01,client_e547b89c05043229,content_89d4bfd4558ec9b9,0.0,117.0,8.223562,0.0,0.0,1.0
1,2026-03-01,client_e547b89c05043229,content_0f7da309daf39bb8,0.0,851.0,15.169861,0.0,0.0,1.0
2,2026-03-01,client_e547b89c05043229,content_8c99c861f5cb1a47,0.0,20.0,16.680556,0.0,0.0,0.0
3,2026-03-01,client_e547b89c05043229,content_6210e5a42848b5fb,5.0,2034.0,3.536536,2.0,0.0,1.0
4,2026-03-01,client_e547b89c05043229,content_8377b3ff6652698c,0.0,579.0,5.197947,0.0,0.0,1.0


In [31]:
print("Feature frame shape:", feature_frame.shape)

Feature frame shape: (9841378, 9)


In [32]:
feature_frame.head()

,report_date,client_hash_id,content_hash_id,prev_7d_gsc_clicks,prev_7d_gsc_impressions,prev_7d_avg_position,prev_7d_organic_sessions,prev_7d_scroll_events,future_7d_gsc_clicks
0,2026-03-01,client_e547b89c05043229,content_89d4bfd4558ec9b9,0.0,117.0,8.223562,0.0,0.0,1.0
1,2026-03-01,client_e547b89c05043229,content_0f7da309daf39bb8,0.0,851.0,15.169861,0.0,0.0,1.0
2,2026-03-01,client_e547b89c05043229,content_8c99c861f5cb1a47,0.0,20.0,16.680556,0.0,0.0,0.0
3,2026-03-01,client_e547b89c05043229,content_6210e5a42848b5fb,5.0,2034.0,3.536536,2.0,0.0,1.0
4,2026-03-01,client_e547b89c05043229,content_8377b3ff6652698c,0.0,579.0,5.197947,0.0,0.0,1.0


In [33]:
print("Rows:", len(feature_frame))
print("Missing label:", feature_frame["future_7d_gsc_clicks"].isna().sum())

Rows: 9841378
Missing label: 1


### Verification 1 — Grain

The intended grain is one client-content pair on one report date. The query compares total rows with the number of distinct `(report_date, client_hash_id, content_hash_id)` combinations and explicitly checks for duplicate grain keys.

In [34]:
grain_verification = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
            AS distinct_grain_rows,
        COUNT(*) - COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
            AS duplicate_grain_rows
    FROM {daily}
""").df()

grain_verification

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,duplicate_grain_rows
0,9841378,9841378,0


### Verification 2 — March row count and date span

This query verifies the size and temporal coverage of the March 2026 development slice.

In [35]:
count_window_verification = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {daily}
""").df()

count_window_verification

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Verification 3 — Data availability
Availability is measured using explicit `IS TRUE` conditions rather than treating non-null values as equivalent to source availability.

In [36]:
availability_verification = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM {daily}
""").df()

availability_verification

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


### Deliberate leakage experiment

To demonstrate label leakage, I intentionally add the future outcome `future_7d_gsc_clicks` to the feature set. This is not a valid feature because it is the quantity being predicted and is unavailable at the decision moment. The purpose is to demonstrate how a quick model score can become artificially strong when outcome information is leaked into the inputs.

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

model_data = feature_frame.dropna(
    subset=[
        "prev_7d_gsc_clicks",
        "prev_7d_gsc_impressions",
        "prev_7d_avg_position",
        "prev_7d_organic_sessions",
        "prev_7d_scroll_events",
        "future_7d_gsc_clicks"
    ]
).copy()

leaky_features = [
    "prev_7d_gsc_clicks",
    "prev_7d_gsc_impressions",
    "prev_7d_avg_position",
    "prev_7d_organic_sessions",
    "prev_7d_scroll_events",
    "future_7d_gsc_clicks"   # DELIBERATE LEAK
]

X = model_data[leaky_features]
y = model_data["future_7d_gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

leaky_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

leaky_score = r2_score(y_test, leaky_pred)

print("Leaky R²:", leaky_score)

Leaky R²: 0.9997433832183086


Honest Model

In [43]:
honest_features = [
    "prev_7d_gsc_clicks",
    "prev_7d_gsc_impressions",
    "prev_7d_avg_position",
    "prev_7d_organic_sessions",
    "prev_7d_scroll_events"
]

X = model_data[honest_features]
y = model_data["future_7d_gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

honest_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = r2_score(y_test, honest_pred)

print("Honest R²:", honest_score)

Honest R²: 0.797250016500885


Remove the leak

In [46]:
print("Leaky R² :", leaky_score)
print("Honest R²:", honest_score)
print("Score inflation from leakage:", leaky_score - honest_score)

Leaky R² : 0.9997433832183086
Honest R²: 0.797250016500885
Score inflation from leakage: 0.20249336671742357


Explicitly delete the leaky column

In [40]:
honest_feature_frame = feature_frame.drop(
    columns=["future_7d_gsc_clicks"]
)

print("Leaky label-derived column removed.")
print("Remaining columns:")
print(honest_feature_frame.columns.tolist())

Leaky label-derived column removed.
Remaining columns:
['report_date', 'client_hash_id', 'content_hash_id', 'prev_7d_gsc_clicks', 'prev_7d_gsc_impressions', 'prev_7d_avg_position', 'prev_7d_organic_sessions', 'prev_7d_scroll_events']


### Leakage result

The deliberate leakage experiment produced an R² of 0.999743, which is suspiciously close to perfect because the target itself (`future_7d_gsc_clicks`) was included among the model inputs.

After removing the label-derived column and evaluating only the five historical features, the honest R² is retained as the meaningful score. The difference demonstrates why a feature must be available at the decision moment and why future outcome information cannot be used as an input.

## 4. Data limits
### Five-feature frame

The feature frame uses only information from the seven days before the decision date. The target is the total GSC clicks during the following seven days.

| Feature | Definition | Available when? |
|---|---|---|
| `prev_7d_gsc_clicks` | Sum of GSC clicks during the previous 7 days | Knowable at the decision moment because the seven-day history has already occurred. |
| `prev_7d_gsc_impressions` | Sum of GSC impressions during the previous 7 days | Knowable at the decision moment because these are historical observations. |
| `prev_7d_avg_position` | Mean GSC average position during the previous 7 days | Knowable at the decision moment because it uses only prior-day observations. |
| `prev_7d_organic_sessions` | Sum of organic sessions during the previous 7 days | Knowable at the decision moment when organic-session data is available. |
| `prev_7d_scroll_events` | Sum of scroll events during the previous 7 days | Knowable at the decision moment when scroll-event data is available. |

**Label:** `future_7d_gsc_clicks` — total GSC clicks during the seven days following the decision date.

### Data limits

- **Unbalanced history:** GSC and GA4 availability are not uniform across rows. In March 2026, GSC data is marked available for 3,611,061 of 9,841,378 rows, while GA4 data is marked available for 413,966 rows.
- **GSC-only early rows:** Some observations have GSC measurements while GA4-related fields are unavailable, limiting comparisons between search and analytics behavior.
- **Window overlaps:** Historical feature windows and future outcome windows use adjacent daily observations, so the feature window must end before the decision date and the label window must begin after it.
- **No causal interpretation:** This warehouse records observed performance and cannot by itself establish that a feature causes future clicks. Results should therefore be treated as directional decision-support.
- **Final-month limitation:** June 2026 is the final available month and should be treated as a sealed outcome/test month rather than used to develop label logic.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.